In [1]:
# 必要なライブラリのインポート
from torch_geometric.datasets import Planetoid

# torch_geometric の Dataset としてダウンロード
dataset = Planetoid(root="./dataset", name="Cora", split="full")

Processing...
Done!


In [2]:
from torch_geometric.transforms import RandomNodeSplit

# ノードを学習データとテストデータに分割
node_splitter = RandomNodeSplit(
    split="train_rest",  # 分割方法
    num_splits=1,  # 分割数
    num_val=0.0,  # 検証データの割合
    num_test=0.4,  # テストデータの割合
    key="y",  # 正解データの属性名
)
splitted_data = node_splitter(dataset._data)
print(splitted_data.node_attrs())
print(splitted_data.train_mask)
print(splitted_data.test_mask)

['test_mask', 'y', 'val_mask', 'x', 'train_mask']
tensor([False, False,  True,  ...,  True,  True, False])
tensor([ True,  True, False,  ..., False, False,  True])


In [3]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, Sequential


# GCNモデルの定義
class GCN(torch.nn.Module):
    def __init__(
        self,
        num_node_features: int,  # 入力層の次元数 = ノードの特徴量の次元数
        projection_dim: int,  # 中間層の次元数
        num_classes: int,  # 出力層の次元数 = 分類先のクラス数
    ) -> None:
        super().__init__()
        self.conv1 = GCNConv(num_node_features, projection_dim)
        self.conv2 = GCNConv(projection_dim, num_classes)

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

In [4]:
# GCNモデルのインスタンス化
device = "cuda" if torch.cuda.is_available() else "cpu"
gcn_model = GCN(
    num_node_features=dataset.num_node_features,
    projection_dim=64,
    num_classes=dataset.num_classes,
).to(device)

# 最適化アルゴリズムの選択
gcn_optimizer = torch.optim.Adam(list(gcn_model.parameters()), lr=0.01)

In [5]:
from tqdm import tqdm


# GCNの学習を行う関数の定義
def train_gcn() -> float:
    gcn_model.train()
    total_loss = 0.0
    gcn_optimizer.zero_grad()
    out = gcn_model(splitted_data)
    loss = F.cross_entropy(
        out[splitted_data.train_mask],
        splitted_data.y[splitted_data.train_mask],
    )
    loss.backward()
    gcn_optimizer.step()
    return loss.item()


# GCNの学習の実行
for epoch in tqdm(range(200)):
    loss = train_gcn()
    print(f"train loss : {loss:.4f}")

  4%|▍         | 9/200 [00:00<00:04, 38.63it/s]

train loss : 1.9533
train loss : 1.7212
train loss : 1.5123
train loss : 1.3116
train loss : 1.1044
train loss : 0.9286
train loss : 0.7883
train loss : 0.6494
train loss : 0.5612
train loss : 0.4953
train loss : 0.4147
train loss : 0.3790
train loss : 0.3479
train loss : 0.3268
train loss : 0.3067
train loss : 0.2813


 12%|█▎        | 25/200 [00:00<00:02, 58.94it/s]

train loss : 0.2720
train loss : 0.2510
train loss : 0.2443
train loss : 0.2260
train loss : 0.2244
train loss : 0.2202
train loss : 0.2022
train loss : 0.2002
train loss : 0.1849
train loss : 0.1877
train loss : 0.1756
train loss : 0.1596
train loss : 0.1613
train loss : 0.1554


 20%|██        | 41/200 [00:00<00:02, 67.99it/s]

...

train loss : 0.0232
train loss : 0.0220
train loss : 0.0243
train loss : 0.0197
train loss : 0.0229
train loss : 0.0191
train loss : 0.0198
train loss : 0.0212
train loss : 0.0189
train loss : 0.0174


 94%|█████████▎| 187/200 [00:02<00:00, 57.02it/s]

train loss : 0.0197
train loss : 0.0193
train loss : 0.0197
train loss : 0.0193
train loss : 0.0204
train loss : 0.0190
train loss : 0.0182
train loss : 0.0175
train loss : 0.0202
train loss : 0.0183
train loss : 0.0213
train loss : 0.0195
train loss : 0.0180
train loss : 0.0178


100%|██████████| 200/200 [00:03<00:00, 62.77it/s]

train loss : 0.0159
train loss : 0.0167
train loss : 0.0194
train loss : 0.0174
train loss : 0.0151
train loss : 0.0197
train loss : 0.0180
train loss : 0.0185
